# PBTA_RNA Clinical Data Analysis

**Exploratory analysis of pediatric brain tumor clinical data.**

This notebook covers:
- Patient demographics and survival outcomes
- Cancer predispositions
- Sample-level tumor annotations
- Cross-dataset integration
- Survival analysis by cancer group and molecular subtype
- Tumor purity and ploidy analysis

Data: PBTA_RNA study

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/home/alon/menow_home_ass/PBTA_RNA"
PATIENT_FILE = f"{DATA_DIR}/data_clinical_patient_attributes.txt"
SAMPLE_FILE = f"{DATA_DIR}/data_clinical_sample_attributes.txt"

def read_patients():
    return pd.read_csv(PATIENT_FILE, sep="\t", header=4,
                       dtype={"AGE": float, "AGE_IN_DAYS": float,
                              "OS_MONTHS": float, "EFS_MONTHS": float})

def read_samples():
    return pd.read_csv(SAMPLE_FILE, sep="\t", header=4)

print("Imports and data paths ready.")


In [ ]:
# --- Helper functions used across steps ---

def clean_os_status(df):
    df = df.copy()
    df["OS_STATUS"] = df["OS_STATUS"].str.strip()
    df["os_label"] = df["OS_STATUS"].str.replace(r"^\d+:", "", regex=True)
    df["os_event"] = df["OS_STATUS"].apply(
        lambda x: 1 if pd.notna(x) and x.startswith("1:") else (0 if pd.notna(x) and x.startswith("0:") else np.nan))
    return df

def clean_efs_status(df):
    df = df.copy()
    df["EFS_STATUS"] = df["EFS_STATUS"].str.strip()
    df["efs_detail"] = df["EFS_STATUS"].str.replace(r"^\d+:", "", regex=True)
    df["efs_event"] = df["EFS_STATUS"].apply(
        lambda x: 0 if pd.notna(x) and x == "0:No Event"
        else (1 if pd.notna(x) and x != "1:NA" else np.nan))
    return df

def clean_race_ethnicity(df):
    df = df.copy()
    df["RACE"] = df["RACE"].fillna("Unknown")
    df["RACE"] = df["RACE"].replace({"Not Reported": "Unknown", "Reported Unknown": "Unknown"})
    df["ETHNICITY"] = df["ETHNICITY"].fillna("Unknown")
    df["ETHNICITY"] = df["ETHNICITY"].replace({"Not Reported": "Unknown", "Reported Unknown": "Unknown"})
    return df

def clean_predispositions(df):
    df = df.copy()
    df["CANCER_PREDISPOSITIONS"] = df["CANCER_PREDISPOSITIONS"].fillna("Unknown")
    df["CANCER_PREDISPOSITIONS"] = df["CANCER_PREDISPOSITIONS"].replace("Not Reported", "Unknown")
    df["CANCER_PREDISPOSITIONS"] = df["CANCER_PREDISPOSITIONS"].replace("None documented", "No predisposition")
    return df

def clean_molecular_subtype(df):
    df = df.copy()
    df["MOLECULAR_SUBTYPE"] = df["MOLECULAR_SUBTYPE"].fillna("Unclassified")
    df["MOLECULAR_SUBTYPE"] = df["MOLECULAR_SUBTYPE"].replace("To be classified", "Unclassified")
    return df

def clean_tumor_fraction_ploidy(df):
    df = df.copy()
    df["TUMOR_FRACTION_GROUP"] = np.where(df["TUMOR_FRACTION"].isna(), "Unknown", "Measured")
    df["TUMOR_PLOIDY_GROUP"] = np.where(df["TUMOR_PLOIDY"].isna(), "Unknown", "Measured")
    return df

def kaplan_meier(times, events):
    df = pd.DataFrame({"time": times, "event": events}).dropna()
    df = df.sort_values("time")
    n = len(df)
    surv = 1.0
    result = []
    n_at_risk = n
    for t, grp in df.groupby("time", sort=False):
        n_events = int(grp["event"].sum())
        if n_events > 0:
            surv *= (1 - n_events / n_at_risk)
        result.append({"time": t, "survival": surv, "n_at_risk": n_at_risk, "n_events": n_events})
        n_at_risk -= len(grp)
    return pd.DataFrame(result)

def add_km(fig, km_df, label, color):
    fig.add_trace(go.Scatter(
        x=km_df["time"], y=km_df["survival"],
        mode="lines", name=label,
        line=dict(color=color, width=2, shape="hv"),
        legendgroup=label,
        hovertemplate=f"Time: %{{x}}<br>Survival: %{{y:.3f}}<br>At risk: %{{text}}<extra>{label}</extra>",
        text=km_df["n_at_risk"]
    ))
    censored = km_df[km_df["n_events"] == 0]
    if len(censored) > 0:
        fig.add_trace(go.Scatter(
            x=censored["time"], y=censored["survival"],
            mode="markers", name=label+" (censored)",
            marker=dict(symbol="line-ns", color=color, size=6, line=dict(width=1)),
            legendgroup=label, showlegend=False
        ))
    return fig

def logrank_test(t1, e1, t2, e2):
    from scipy.stats import chi2
    all_t = sorted(set(pd.concat([pd.Series(t1.dropna()), pd.Series(t2.dropna())]).dropna()))
    if len(all_t) < 2:
        return 1.0
    o1e = 0; v = 0
    d1 = pd.DataFrame({"time": t1, "event": e1}).dropna()
    d2 = pd.DataFrame({"time": t2, "event": e2}).dropna()
    n1 = len(d1); n2 = len(d2)
    for t in all_t:
        r1 = (d1["time"] >= t).sum()
        r2 = (d2["time"] >= t).sum()
        nr = r1 + r2
        if nr == 0: continue
        o1 = int(((d1["time"] == t) & (d1["event"] == 1)).sum())
        o2 = int(((d2["time"] == t) & (d2["event"] == 1)).sum())
        ot = o1 + o2
        if ot == 0: continue
        e1 = ot * r1 / nr
        o1e += (o1 - e1)
        if nr > 1:
            v += ot * (r1 / nr) * (r2 / nr) * (nr - ot) / (nr - 1)
    if v <= 0: return 1.0
    return 1 - chi2.cdf(o1e**2 / v, 1)

def multi_logrank(groups):
    from scipy.stats import chi2
    import numpy as np
    ng = len(groups)
    if ng < 2: return 1.0
    all_t = sorted(set(pd.concat([pd.Series(g[0].dropna()) for g in groups]).dropna()))
    if len(all_t) < 2: return 1.0
    O = np.zeros(ng); E = np.zeros(ng); V = np.zeros((ng, ng))
    for t in all_t:
        ar = np.array([(g[0] >= t).sum() for g in groups])
        nr = ar.sum()
        if nr == 0: continue
        ev = np.array([int((g[0] == t) & (g[1] == 1)) for g in groups])
        ot = ev.sum()
        if ot == 0: continue
        O += ev; E += ot * ar / nr
        if nr > 1:
            for i in range(ng):
                for j in range(ng):
                    if i == j:
                        V[i,j] += ot * ar[i]/nr * (1-ar[i]/nr) * (nr-ot)/(nr-1)
                    else:
                        V[i,j] -= ot * ar[i]/nr * ar[j]/nr * (nr-ot)/(nr-1)
    try:
        chi2s = (O-E) @ np.linalg.pinv(V) @ (O-E)
        return 1 - chi2.cdf(chi2s, ng-1)
    except:
        return 1.0

print("Helper functions loaded.")


## Step 1: Load & Profile Patient Data

**Purpose:** Comprehensive overview of the patient dataset — size, column types, missingness levels.

In [ ]:
df = read_patients()
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print()
summary_rows = []
for col in df.columns:
    n_non = df[col].notna().sum()
    n_nul = df[col].isna().sum()
    pct = n_nul / len(df) * 100
    if df[col].dtype == "object":
        extra = f"unique={df[col].nunique()}"
    else:
        extra = f"min={df[col].min()}, max={df[col].max()}"
    summary_rows.append({"Column": col, "Dtype": str(df[col].dtype),
                         "Non-null": n_non, "Null": n_nul,
                         "% Null": f"{pct:.1f}%", "Extra": extra})
print(pd.DataFrame(summary_rows).to_string(index=False))


In [ ]:
miss = df.isna().mean().sort_values(ascending=False) * 100
fig = px.bar(x=miss.index, y=miss.values,
             title="Patient Data -- Missingness by Column",
             labels={"x": "Column", "y": "% Missing"},
             text=[f"{v:.1f}%" for v in miss.values])
fig.update_traces(marker_color="crimson", textposition="outside")
fig.update_layout(xaxis_tickangle=-45, height=450)
fig.show()


In [ ]:
# Validation: Missing values
AGE_nulls = df["AGE"].isna().sum()
print(f"AGE has {AGE_nulls} missing values ({AGE_nulls/len(df):.1%}) -- these will be excluded from age-specific plots")
print(f"OS_STATUS null: {df["OS_STATUS"].isna().sum()}")
print(f"EFS_STATUS null: {df["EFS_STATUS"].isna().sum()}")


## Step 2: Patient Demographics

**Purpose:** Age distribution, sex balance, race/ethnicity makeup.

In [ ]:
df = read_patients()
df = clean_os_status(df)
df = clean_efs_status(df)
df = clean_race_ethnicity(df)
age_data = df.dropna(subset=["AGE"])
n_sex = age_data["SEX"].nunique()
n_race = age_data["RACE"].nunique()
n_eth = age_data["ETHNICITY"].nunique()

fig_age = go.Figure()
fig_age.add_trace(go.Histogram(x=age_data["AGE"], nbinsx=40, name="All Patients",
                               marker_color="steelblue", opacity=0.75))
colors = px.colors.qualitative.Plotly
# Sex traces
for i, cat in enumerate(age_data["SEX"].value_counts().index):
    d = age_data[age_data["SEX"] == cat]
    fig_age.add_trace(go.Histogram(x=d["AGE"], nbinsx=40, name=f"Sex: {cat}",
                                   marker_color=colors[i], opacity=0.6, visible=False))
# Race traces
for i, cat in enumerate(age_data["RACE"].value_counts().index):
    d = age_data[age_data["RACE"] == cat]
    fig_age.add_trace(go.Histogram(x=d["AGE"], nbinsx=40, name=f"Race: {cat}",
                                   marker_color=colors[i%len(colors)], opacity=0.6, visible=False))
# Ethnicity traces
for i, cat in enumerate(age_data["ETHNICITY"].value_counts().index):
    d = age_data[age_data["ETHNICITY"] == cat]
    fig_age.add_trace(go.Histogram(x=d["AGE"], nbinsx=40, name=f"Ethnicity: {cat}",
                                   marker_color=colors[i%len(colors)], opacity=0.6, visible=False))

all_n = 1 + n_sex + n_race + n_eth
fig_age.update_layout(
    updatemenus=[dict(
        buttons=[
            dict(label="Overall", method="update",
                 args=[{"visible": [True]+[False]*(all_n-1)},
                       {"title": "Age Distribution -- Overall", "barmode": "overlay"}]),
            dict(label="By SEX", method="update",
                 args=[{"visible": [True]+[True]*n_sex+[False]*(n_race+n_eth)},
                       {"title": "Age Distribution by SEX", "barmode": "overlay"}]),
            dict(label="By RACE", method="update",
                 args=[{"visible": [True]+[False]*n_sex+[True]*n_race+[False]*n_eth},
                       {"title": "Age Distribution by RACE", "barmode": "overlay"}]),
            dict(label="By ETHNICITY", method="update",
                 args=[{"visible": [True]+[False]*n_sex+[False]*n_race+[True]*n_eth},
                       {"title": "Age Distribution by ETHNICITY", "barmode": "overlay"}]),
        ], direction="down", showactive=True, x=1.0, y=1.15
    )],
    title="Age Distribution -- Overall",
    xaxis_title="Age (years)", yaxis_title="Count",
    height=500, bargap=0.05
)
fig_age.show()


In [ ]:
# 2x2 grid: SEX, RACE, ETHNICITY + Age stats
fig_demo = make_subplots(rows=2, cols=2,
    subplot_titles=("Sex Distribution", "Race Distribution",
                    "Ethnicity Distribution", "Age Summary Stats"),
    specs=[[{"type":"bar"},{"type":"bar"}],[{"type":"bar"},{"type":"table"}]])
sex_c = df["SEX"].fillna("NaN").value_counts().reset_index()
sex_c.columns = ["SEX","count"]
fig_demo.add_trace(go.Bar(x=sex_c["SEX"], y=sex_c["count"],
    marker_color="lightblue", text=sex_c["count"], textposition="outside", showlegend=False), row=1, col=1)
race_c = df["RACE"].value_counts().reset_index()
race_c.columns = ["RACE","count"]
fig_demo.add_trace(go.Bar(x=race_c["RACE"], y=race_c["count"],
    marker_color="lightgreen", text=race_c["count"], textposition="outside", showlegend=False), row=1, col=2)
eth_c = df["ETHNICITY"].value_counts().reset_index()
eth_c.columns = ["ETHNICITY","count"]
fig_demo.add_trace(go.Bar(x=eth_c["ETHNICITY"], y=eth_c["count"],
    marker_color="lightskyblue", text=eth_c["count"], textposition="outside", showlegend=False), row=2, col=1)
age_cl = df["AGE"].dropna()
age_st = pd.DataFrame([
    ["Count",str(len(age_cl))],["Mean",f"{age_cl.mean():.1f}"],["Median",f"{age_cl.median():.1f}"],
    ["Std",f"{age_cl.std():.1f}"],["Min",f"{age_cl.min():.1f}"],["Max",f"{age_cl.max():.1f}"],
    ["Missing",str(df["AGE"].isna().sum())]], columns=["Stat","Value"])
fig_demo.add_trace(go.Table(header=dict(values=["Stat","Value"],fill_color="lightblue",align="left"),
    cells=dict(values=[age_st["Stat"],age_st["Value"]],align="left",height=25)), row=2, col=2)
fig_demo.update_layout(height=600, title_text="Patient Demographics Overview")
fig_demo.show()


In [ ]:
# Validation
print(f"RACE unique after: {sorted(df["RACE"].unique())}")
print(f"ETHNICITY unique after: {sorted(df["ETHNICITY"].unique())}")
print(f"Patients with AGE: {df["AGE"].notna().sum()} / {len(df)}")


## Step 3: Patient Survival Overview

**Purpose:** Outcome overview and Kaplan-Meier survival curves.

In [ ]:
df = read_patients()
df = clean_os_status(df)
df = clean_efs_status(df)
# OS pie
os_c = df["os_label"].value_counts(dropna=False).reset_index()
os_c.columns = ["Status","Count"]
os_c["Status"] = os_c["Status"].fillna("Unknown")
fig_osp = px.pie(os_c, values="Count", names="Status",
    title="Overall Survival Status", hole=0.3)
fig_osp.update_traces(textinfo="label+percent")
fig_osp.show()
# EFS: binary + detailed
df["efs_binary"] = df["efs_event"].map({1:"Event",0:"No Event"}).fillna("Unknown")
fig_efs = make_subplots(rows=1, cols=2,
    subplot_titles=("EFS Binary","EFS Detailed"))
eb = df["efs_binary"].value_counts().reset_index()
eb.columns = ["Status","Count"]
fig_efs.add_trace(go.Bar(x=eb["Status"], y=eb["Count"],
    marker_color=["lightcoral","lightgreen","lightgray"],
    text=eb["Count"], textposition="outside", showlegend=False), row=1, col=1)
ed = df["efs_detail"].value_counts(dropna=False).reset_index()
ed.columns = ["Status","Count"]
ed["Status"] = ed["Status"].fillna("Unknown")
fig_efs.add_trace(go.Bar(x=ed["Status"], y=ed["Count"],
    marker_color="lightcoral", text=ed["Count"],
    textposition="outside", showlegend=False), row=1, col=2)
fig_efs.update_layout(title="Event-Free Survival Status", height=450, xaxis2_tickangle=-45)
fig_efs.show()


In [ ]:
# KM curves
fig_km = make_subplots(rows=2, cols=1,
    subplot_titles=("Overall Survival -- KM","Event-Free Survival -- KM"),
    vertical_spacing=0.15)
# OS KM
os_c = df[["OS_MONTHS","os_event"]].dropna()
km_os = kaplan_meier(os_c["OS_MONTHS"], os_c["os_event"])
fig_km = add_km(fig_km, km_os, "OS", "darkblue")
# EFS KM
efs_c = df[["EFS_MONTHS","efs_event"]].dropna()
km_efs = kaplan_meier(efs_c["EFS_MONTHS"], efs_c["efs_event"])
fig_km.add_trace(go.Scatter(
    x=km_efs["time"], y=km_efs["survival"],
    mode="lines", name="EFS",
    line=dict(color="darkred", width=2, shape="hv"),
    hovertemplate="Time: %{x}<br>Survival: %{y:.3f}<extra>EFS</extra>"
), row=2, col=1)
fig_km.update_layout(height=600, title_text="Kaplan-Meier Survival Curves")
fig_km.update_yaxes(range=[-0.05, 1.05])
fig_km.show()


In [ ]:
# Validation
co = df[["OS_MONTHS","os_event"]].dropna()
ce = df[["EFS_MONTHS","efs_event"]].dropna()
print(f"OS: {len(co)}/{len(df)} have complete (time, status)")
print(f"EFS: {len(ce)}/{len(df)} have complete (time, status)")
print(f"OS status:\n{df["OS_STATUS"].value_counts(dropna=False)}")
print(f"\nEFS binary:\n{df["efs_event"].value_counts(dropna=False)}")
print(f"EFS detailed:\n{df["efs_detail"].value_counts(dropna=False)}")


## Step 4: Cancer Predispositions -- Prevalence & Demographics

**Purpose:** How common each predisposition is, and demographic patterns across predispositions.

In [ ]:
import re
df = read_patients()
df = clean_predispositions(df)
df = clean_race_ethnicity(df)
def explode_predispositions(df):
    rows = []
    for _, row in df.iterrows():
        pred = row["CANCER_PREDISPOSITIONS"]
        if pred in ("Unknown", "No predisposition"):
            rows.append({**row, "pred_exploded": pred})
        elif ")," in str(pred):
            parts = re.split(r"\),\s*", str(pred))
            for i, p in enumerate(parts):
                if i < len(parts)-1:
                    p = p + ")"
                rows.append({**row, "pred_exploded": p.strip()})
        else:
            rows.append({**row, "pred_exploded": pred})
    return pd.DataFrame(rows)
df_ex = explode_predispositions(df)
print(f"Patients before explosion: {df["PATIENT_ID"].nunique()}")
print(f"Rows after explosion: {len(df_ex)}")
multi_p = df[df["CANCER_PREDISPOSITIONS"].str.contains(r"\),", na=False, regex=True)]
print(f"Multi-syndrome patients: {len(multi_p)}")


In [ ]:
# Prevalence bar
total_p = df["PATIENT_ID"].nunique()
pred_counts = df_ex[~df_ex["pred_exploded"].isin(["No predisposition","Unknown"])]
pred_counts = pred_counts["pred_exploded"].value_counts().head(15).reset_index()
pred_counts.columns = ["Predisposition","Count"]
pred_counts["% of Patients"] = (pred_counts["Count"]/total_p*100).round(1)
fig_pred = px.bar(pred_counts, y="Predisposition", x="Count", orientation="h",
    title=f"Top 15 Cancer Predispositions (N={total_p} patients)",
    text=[f"{c} ({p:.1f}%)" for c,p in zip(pred_counts["Count"],pred_counts["% of Patients"])],
    color="Count", color_continuous_scale="Blues")
fig_pred.update_traces(textposition="outside")
fig_pred.update_layout(height=500, yaxis={"categoryorder":"total ascending"})
fig_pred.show()


In [ ]:
# Interactive explorer - show top 10
top10 = pred_counts.head(10)
fig_ex = make_subplots(rows=2, cols=2,
    subplot_titles=("Age: With vs Without","Sex Breakdown","Prevalence %","Summary Table"),
    specs=[[{"type":"box"},{"type":"bar"}],[{"type":"bar"},{"type":"table"}]])
# Default: first predisposition
dp = top10["Predisposition"].iloc[0]
with_p = df_ex[df_ex["pred_exploded"]==dp]
without_p = df[~df["PATIENT_ID"].isin(with_p["PATIENT_ID"].unique())]
fig_ex.add_trace(go.Box(y=with_p["AGE"].dropna(), name="With",
    marker_color=colors[0]), row=1, col=1)
fig_ex.add_trace(go.Box(y=without_p["AGE"].dropna(), name="Without",
    marker_color="lightgray"), row=1, col=1)
sx = with_p["SEX"].fillna("NaN").value_counts().reset_index()
sx.columns = ["SEX","count"]
fig_ex.add_trace(go.Bar(x=sx["SEX"], y=sx["count"],
    marker_color="lightcoral", text=sx["count"],
    textposition="outside", showlegend=False), row=1, col=2)
fig_ex.add_trace(go.Bar(x=top10["Predisposition"], y=top10["% of Patients"],
    marker_color="steelblue", text=top10["% of Patients"],
    textposition="outside", showlegend=False), row=2, col=1)
t10d = top10.rename(columns={"% of Patients":"Pct"})
fig_ex.add_trace(go.Table(
    header=dict(values=list(t10d.columns),fill_color="lightblue",align="left"),
    cells=dict(values=[t10d[c] for c in t10d.columns],align="left",height=22)), row=2, col=2)
fig_ex.update_layout(height=650, title_text="Predisposition Explorer (Top 10)")
fig_ex.show()


In [ ]:
# Summary table
ps = df_ex[~df_ex["pred_exploded"].isin(["No predisposition","Unknown"])]
ps = ps.groupby("pred_exploded").agg(
    Count=("PATIENT_ID","nunique"),
    MedianAge=("AGE","median"),
    PctFemale=("SEX",lambda x: (x=="Female").sum()/len(x)*100 if len(x)>0 else 0)
).reset_index().sort_values("Count",ascending=False)
ps["% of Patients"] = (ps["Count"]/total_p*100).round(1)
ps.columns = ["Predisposition","Count","Median Age","% Female","% of Patients"]
print(ps.to_string(index=False))
print(f"\nTotal patients: {total_p}")


## Step 5: Load & Profile Sample Data

**Purpose:** Overview of the sample-level dataset.

In [ ]:
df = read_samples()
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")
rows = []
for col in df.columns:
    nn = df[col].notna().sum(); nnu = df[col].isna().sum()
    p = nnu/len(df)*100
    if df[col].dtype=="object": ex = f"unique={df[col].nunique()}"
    else: ex = f"min={df[col].min():.2f}, max={df[col].max():.2f}"
    rows.append({"Column":col,"Dtype":str(df[col].dtype),"Non-null":nn,"Null":nnu,"% Null":f"{p:.1f}%","Extra":ex})
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
miss = df.isna().mean().sort_values(ascending=False)*100
fig = px.bar(x=miss.index, y=miss.values,
    title="Sample Data -- Missingness by Column",
    labels={"x":"Column","y":"% Missing"},
    text=[f"{v:.1f}%" for v in miss.values],
    color=miss.values, color_continuous_scale="Reds")
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()
print("\nMissing > 0:")
print(miss[miss>0].round(1).to_string())


## Step 6: Sample Cancer Type Distributions

**Purpose:** Histological and anatomical breakdown of all samples.

In [ ]:
df = read_samples()
fig_ct = make_subplots(rows=2, cols=2,
    subplot_titles=("Broad Histology","Cancer Group","CNS Region","Tumor Type"),
    specs=[[{"type":"bar"},{"type":"bar"}],[{"type":"bar"},{"type":"bar"}]])
hc = df["BROAD_HISTOLOGY"].value_counts().head(12).reset_index()
hc.columns = ["H","C"]
fig_ct.add_trace(go.Bar(x=hc["H"], y=hc["C"], marker_color="lightblue",
    text=hc["C"], textposition="outside", showlegend=False), row=1, col=1)
cg = df["CANCER_GROUP"].value_counts().head(12).reset_index()
cg.columns = ["H","C"]
fig_ct.add_trace(go.Bar(x=cg["H"], y=cg["C"], marker_color="lightgreen",
    text=cg["C"], textposition="outside", showlegend=False), row=1, col=2)
cr = df["CNS_REGION"].value_counts().reset_index()
cr.columns = ["H","C"]
fig_ct.add_trace(go.Bar(x=cr["H"], y=cr["C"], marker_color="lightsalmon",
    text=cr["C"], textposition="outside", showlegend=False), row=2, col=1)
tt = df["TUMOR_TYPE"].value_counts().reset_index()
tt.columns = ["H","C"]
rare = tt[tt["C"]<30]; comm = tt[tt["C"]>=30]
if len(rare)>0:
    other = pd.DataFrame([["Other",rare["C"].sum()]],columns=["H","C"])
    tt_p = pd.concat([comm,other],ignore_index=True)
else: tt_p = comm
fig_ct.add_trace(go.Bar(x=tt_p["H"], y=tt_p["C"], marker_color="plum",
    text=tt_p["C"], textposition="outside", showlegend=False), row=2, col=2)
fig_ct.update_layout(title_text="Sample Cancer Type Distributions", height=650,
    xaxis_tickangle=-45, xaxis2_tickangle=-45, xaxis3_tickangle=-45, xaxis4_tickangle=-45)
fig_ct.show()


In [ ]:
print(f"Most common CG: {df["CANCER_GROUP"].value_counts().index[0]} ({df["CANCER_GROUP"].value_counts().iloc[0]})")
print(f"Total CGs: {df["CANCER_GROUP"].nunique()}")


## Step 7: Tumor Purity & Ploidy

**Purpose:** Distribution of tumor purity and ploidy across samples.

In [ ]:
df = read_samples()
df = clean_tumor_fraction_ploidy(df)
fig_p = make_subplots(rows=1, cols=3,
    subplot_titles=("Tumor Fraction","Tumor Ploidy","Fraction vs Ploidy"),
    specs=[[{"type":"histogram"},{"type":"histogram"},{"type":"scatter"}]])
tf = df[df["TUMOR_FRACTION_GROUP"]=="Measured"]["TUMOR_FRACTION"].dropna()
fig_p.add_trace(go.Histogram(x=tf, nbinsx=40, marker_color="steelblue", opacity=0.75), row=1, col=1)
uk_tf = (df["TUMOR_FRACTION_GROUP"]=="Unknown").sum()
fig_p.add_annotation(text=f"Unknown: {uk_tf} ({uk_tf/len(df):.1%})",
    xref="paper", yref="paper", x=0.5, y=-0.3, showarrow=False, font=dict(color="gray",size=11),
    row=1, col=1)
tp = df[df["TUMOR_PLOIDY_GROUP"]=="Measured"]["TUMOR_PLOIDY"].dropna()
fig_p.add_trace(go.Histogram(x=tp, nbinsx=30, marker_color="darkgreen", opacity=0.75), row=1, col=2)
uk_tp = (df["TUMOR_PLOIDY_GROUP"]=="Unknown").sum()
fig_p.add_annotation(text=f"Unknown: {uk_tp} ({uk_tp/len(df):.1%})",
    xref="paper", yref="paper", x=0.5, y=-0.3, showarrow=False, font=dict(color="gray",size=11),
    row=1, col=2)
sc = df.dropna(subset=["TUMOR_FRACTION","TUMOR_PLOIDY","CANCER_GROUP"])
top_cg = sc["CANCER_GROUP"].value_counts().head(8).index
sc_p = sc[sc["CANCER_GROUP"].isin(top_cg)]
for cg in top_cg:
    s = sc_p[sc_p["CANCER_GROUP"]==cg]
    fig_p.add_trace(go.Scatter(x=s["TUMOR_FRACTION"], y=s["TUMOR_PLOIDY"],
        mode="markers", name=cg, marker=dict(size=5,opacity=0.6)), row=1, col=3)
fig_p.update_layout(height=450, title_text="Tumor Purity and Ploidy",
    xaxis3_title="Tumor Fraction", yaxis3_title="Tumor Ploidy")
fig_p.show()


In [ ]:
print(f"TUMOR_FRACTION missing: {df["TUMOR_FRACTION"].isna().sum()}/{len(df)}")
print(f"TUMOR_PLOIDY missing: {df["TUMOR_PLOIDY"].isna().sum()}/{len(df)}")
print(f"TF range: {tf.min():.3f} - {tf.max():.3f}")
print(f"TP range: {tp.min():.1f} - {tp.max():.1f}")


## Step 8: Molecular Subtype Landscape

**Purpose:** The diversity of molecular subtypes and their relationship to cancer groups.

In [ ]:
df = read_samples()
df = clean_molecular_subtype(df)
sc = df["MOLECULAR_SUBTYPE"].value_counts().head(20).reset_index()
sc.columns = ["Subtype","Count"]
sc["%"] = (sc["Count"]/len(df)*100).round(1)
fig_s = px.bar(sc, y="Subtype", x="Count", orientation="h",
    title="Top 20 Molecular Subtypes",
    text=[f"{c} ({p:.1f}%)" for c,p in zip(sc["Count"],sc["%"])],
    color="Count", color_continuous_scale="Viridis")
fig_s.update_traces(textposition="outside")
fig_s.update_layout(height=600, yaxis={"categoryorder":"total ascending"})
fig_s.show()


In [ ]:
# Heatmap: top 15 subtypes x top 10 cancer groups
t15 = df["MOLECULAR_SUBTYPE"].value_counts().head(15).index
t10 = df["CANCER_GROUP"].value_counts().head(10).index
ct = pd.crosstab(df["MOLECULAR_SUBTYPE"], df["CANCER_GROUP"])
ct_f = ct.loc[ct.index.intersection(t15), ct.columns.intersection(t10)]
ct_n = ct_f.div(ct_f.sum(axis=1), axis=0).fillna(0)
fig_h = go.Figure(data=go.Heatmap(
    z=ct_n.values, x=ct_n.columns, y=ct_n.index,
    text=ct_f.values, texttemplate="%{text}", textfont=dict(size=9),
    colorscale="Blues", colorbar=dict(title="Proportion"),
    hovertemplate="Subtype: %{y}<br>Group: %{x}<br>Count: %{text}<br>Prop: %{z:.2f}<extra></extra>"))
fig_h.update_layout(title="Subtype x Cancer Group (Row % + Counts)",
    xaxis_tickangle=-45, height=550, yaxis=dict(autorange="reversed"))
fig_h.show()
STEP8_CT = ct_f
STEP8_TOP = t15
print("Heatmap stored for Step 13 reference.")


In [ ]:
unc = (df["MOLECULAR_SUBTYPE"]=="Unclassified").sum()
print(f"Unclassified: {unc} ({unc/len(df):.1%})")
print(f"Distinct subtypes: {df["MOLECULAR_SUBTYPE"].nunique()}")


## Step 9: Sequencing Strategy & RNA Library

**Purpose:** What sequencing methods and library prep were used.

In [ ]:
df = read_samples()
fig_sq = make_subplots(rows=1, cols=2,
    subplot_titles=("Experiment Strategy","RNA Library Selection"))
es = df["EXPERIMENT_STRATEGY"].value_counts().reset_index()
es.columns = ["S","C"]
fig_sq.add_trace(go.Bar(x=es["S"], y=es["C"], marker_color="teal",
    text=es["C"], textposition="outside", showlegend=False), row=1, col=1)
lb = df["RNA_LIBRARY_SELECTION"].value_counts().reset_index()
lb.columns = ["S","C"]
fig_sq.add_trace(go.Bar(x=lb["S"], y=lb["C"], marker_color="purple",
    text=lb["C"], textposition="outside", showlegend=False), row=1, col=2)
fig_sq.update_layout(title_text="Sequencing Methods", height=400,
    xaxis_tickangle=-45, xaxis2_tickangle=-45)
fig_sq.show()


In [ ]:
print(f"Strategy unique: {df["EXPERIMENT_STRATEGY"].nunique()}")
print(f"Library unique: {df["RNA_LIBRARY_SELECTION"].nunique()}")
print("Strategies:", list(df["EXPERIMENT_STRATEGY"].unique()))


## Step 9a: Multi-Cancer-Group Analysis

**Purpose:** How many patients have samples in different cancer groups.

In [ ]:
df = read_samples()
pg = df.groupby("PATIENT_ID")["CANCER_GROUP"].apply(set).reset_index()
pg["n"] = pg["CANCER_GROUP"].apply(len)
fig_m = make_subplots(rows=1, cols=2,
    subplot_titles=("Groups per Patient","Multi-Group Patients"),
    specs=[[{"type":"bar"},{"type":"table"}]])
gc = pg["n"].value_counts().sort_index().reset_index()
gc.columns = ["G","C"]
fig_m.add_trace(go.Bar(x=gc["G"].astype(str), y=gc["C"],
    marker_color="steelblue", text=gc["C"],
    textposition="outside", showlegend=False), row=1, col=1)
mp = pg[pg["n"]>1].copy()
mp["Groups"] = mp["CANCER_GROUP"].apply(lambda x: ", ".join(sorted(x)))
mp = mp.sort_values("n",ascending=False).head(20)
if len(mp)>0:
    t = mp[["PATIENT_ID","n","Groups"]].head(15)
    fig_m.add_trace(go.Table(
        header=dict(values=["Patient","N","Groups"],fill_color="lightblue",align="left",font=dict(size=10)),
        cells=dict(values=[t["PATIENT_ID"],t["n"],t["Groups"]],align="left",height=22,font=dict(size=9))),
        row=1, col=2)
fig_m.update_layout(height=500, title_text="Multi-Cancer-Group Analysis")
fig_m.show()


In [ ]:
# Co-occurrence for patients with exactly 2 groups
m2 = pg[pg["n"]==2].copy()
if len(m2)>0:
    m2["gl"] = m2["CANCER_GROUP"].apply(lambda x: sorted(x))
    m2["g1"] = m2["gl"].apply(lambda x: x[0])
    m2["g2"] = m2["gl"].apply(lambda x: x[1])
    co = pd.crosstab(m2["g1"], m2["g2"])
    fig_co = go.Figure(data=go.Heatmap(
        z=co.values, x=co.columns, y=co.index,
        text=co.values, texttemplate="%{text}",
        colorscale="Blues"))
    fig_co.update_layout(title="Cancer Group Co-occurrence (2 groups)",
        xaxis_tickangle=-45, height=450)
    fig_co.show()
else: print("No patients with exactly 2 groups.")


In [ ]:
print(f"Patients with >1 CG: {(pg["n"]>1).sum()} / {pg["PATIENT_ID"].nunique()}")
print(f"Max groups: {pg["n"].max()}")
if len(m2)>0: print(f"Exactly 2: {len(m2)}")


## Step 10: Merge Patient + Sample Data

**Purpose:** How well the two datasets connect.

In [ ]:
patients = read_patients()
samples = read_samples()
merged = samples.merge(patients, on="PATIENT_ID", how="left", suffixes=("","_p"))
print(f"Merged: {merged.shape}")
print(f"Samples: {len(samples)}, Patients: {len(patients)}")
sp = set(samples["PATIENT_ID"].unique())
pp = set(patients["PATIENT_ID"].unique())
orph_s = sp - pp
orph_p = pp - sp
print(f"Orphan samples: {len(orph_s)}")
print(f"Orphan patients: {len(orph_p)}")


In [ ]:
# Validation: PATIENT_ID matching
sp = set(samples["PATIENT_ID"].unique())
pp = set(patients["PATIENT_ID"].unique())
print(f"Samples with no matching patient: {len(sp-pp)}")
print(f"Patients with no matching sample: {len(pp-sp)}")
print("(AGE NaN does NOT indicate a merge problem)")


## Step 11: Samples per Patient

**Purpose:** How many patients have single vs. multiple samples.

In [ ]:
samples = read_samples()
patients = read_patients()
merged = samples.merge(patients, on="PATIENT_ID", how="left", suffixes=("","_p"))
sc = merged.groupby("PATIENT_ID")["SAMPLE_ID"].nunique().reset_index()
sc.columns = ["PATIENT_ID","N"]
fig_sp = go.Figure()
fig_sp.add_trace(go.Histogram(x=sc["N"], nbinsx=30,
    marker_color="darkorange", opacity=0.8))
fig_sp.update_layout(title="Samples per Patient",
    xaxis_title="Samples", yaxis_title="Patients", height=400)
fig_sp.show()
print("Top 10 patients:")
print(sc.sort_values("N",ascending=False).head(10).to_string(index=False))


In [ ]:
multi = (sc["N"]>1).sum()
print(f"Patients with >1 sample: {multi} ({multi/len(sc):.1%})")
print(f"Mean samples/patient: {sc["N"].mean():.2f}")


## Step 12: Survival by Cancer Group

**Purpose:** OS and EFS stratified by major cancer groups.

In [ ]:
patients = read_patients()
patients = clean_os_status(patients)
patients = clean_efs_status(patients)
samples = read_samples()
merged = samples.merge(patients, on="PATIENT_ID", how="left", suffixes=("","_p"))
top6 = merged["CANCER_GROUP"].value_counts().head(6).index.tolist()
fig = make_subplots(rows=2, cols=1,
    subplot_titles=("OS by Cancer Group","EFS by Cancer Group"),
    vertical_spacing=0.15)
colors = px.colors.qualitative.Set1
os_data = []; efs_data = []
for i, cg in enumerate(top6):
    sub = merged[merged["CANCER_GROUP"]==cg]
    os_s = sub[["OS_MONTHS","os_event"]].dropna()
    if len(os_s)>5:
        km = kaplan_meier(os_s["OS_MONTHS"], os_s["os_event"])
        fig = add_km(fig, km, cg, colors[i%len(colors)])
        os_data.append((os_s["OS_MONTHS"], os_s["os_event"]))
    efs_s = sub[["EFS_MONTHS","efs_event"]].dropna()
    if len(efs_s)>5:
        km = kaplan_meier(efs_s["EFS_MONTHS"], efs_s["efs_event"])
        fig.add_trace(go.Scatter(
            x=km["time"], y=km["survival"],
            mode="lines", name=cg,
            line=dict(color=colors[i%len(colors)],width=2,shape="hv"),
            legendgroup=cg, showlegend=False), row=2, col=1)
        efs_data.append((efs_s["EFS_MONTHS"], efs_s["efs_event"]))
if len(os_data)>=2:
    pos = multi_logrank(os_data); pef = multi_logrank(efs_data)
    fig.add_annotation(xref="paper",yref="paper",x=0.5,y=1.02,
        text=f"Log-rank OS: p={pos:.4f}",showarrow=False,font=dict(size=11,color="darkblue"),row=1,col=1)
    fig.add_annotation(xref="paper",yref="paper",x=0.5,y=1.02,
        text=f"Log-rank EFS: p={pef:.4f}",showarrow=False,font=dict(size=11,color="darkred"),row=2,col=1)
fig.update_layout(height=650, title_text="Survival by Cancer Group")
fig.update_yaxes(range=[-0.05,1.05])
fig.show()


In [ ]:
# Summary table
rows = []
for cg in top6:
    s = merged[merged["CANCER_GROUP"]==cg]
    os_s = s[["OS_MONTHS","os_event"]].dropna()
    ef_s = s[["EFS_MONTHS","efs_event"]].dropna()
    rows.append({"CG":cg,"N_samples":len(s),"N_patients":s["PATIENT_ID"].nunique(),
        "MedOS":f"{os_s["OS_MONTHS"].median():.1f}" if len(os_s)>0 else "N/A",
        "MedEFS":f"{ef_s["EFS_MONTHS"].median():.1f}" if len(ef_s)>0 else "N/A"})
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
mg = merged.groupby("PATIENT_ID")["CANCER_GROUP"].nunique()
m = mg[mg>1]
print(f"Patients in multiple curves: {len(m)}")
print(f"Total curve entries: {int(m.sum())}")


## Step 13: Survival by Molecular Subtype (Global)

**Purpose:** OS and EFS stratified by molecular subtype.

In [ ]:
patients = read_patients()
patients = clean_os_status(patients)
patients = clean_efs_status(patients)
samples = read_samples()
samples = clean_molecular_subtype(samples)
merged = samples.merge(patients, on="PATIENT_ID", how="left", suffixes=("","_p"))
top6 = merged["MOLECULAR_SUBTYPE"].value_counts().head(6).index.tolist()
fig = make_subplots(rows=2, cols=1,
    subplot_titles=("OS by Molecular Subtype","EFS by Molecular Subtype"),
    vertical_spacing=0.15)
colors = px.colors.qualitative.Set1 + px.colors.qualitative.Set2
os_d = []; ef_d = []
for i, st in enumerate(top6):
    sub = merged[merged["MOLECULAR_SUBTYPE"]==st]
    os_s = sub[["OS_MONTHS","os_event"]].dropna()
    if len(os_s)>5:
        km = kaplan_meier(os_s["OS_MONTHS"], os_s["os_event"])
        fig = add_km(fig, km, st, colors[i%len(colors)])
        os_d.append((os_s["OS_MONTHS"], os_s["os_event"]))
    ef_s = sub[["EFS_MONTHS","efs_event"]].dropna()
    if len(ef_s)>5:
        km = kaplan_meier(ef_s["EFS_MONTHS"], ef_s["efs_event"])
        fig.add_trace(go.Scatter(
            x=km["time"], y=km["survival"],
            mode="lines", name=st,
            line=dict(color=colors[i%len(colors)],width=2,shape="hv"),
            legendgroup=st, showlegend=False), row=2, col=1)
        ef_d.append((ef_s["EFS_MONTHS"], ef_s["efs_event"]))
if len(os_d)>=2:
    pos = multi_logrank(os_d); pef = multi_logrank(ef_d)
    fig.add_annotation(xref="paper",yref="paper",x=0.5,y=1.02,
        text=f"Log-rank OS: p={pos:.4f}",showarrow=False,font=dict(size=11,color="darkblue"),row=1,col=1)
    fig.add_annotation(xref="paper",yref="paper",x=0.5,y=1.02,
        text=f"Log-rank EFS: p={pef:.4f}",showarrow=False,font=dict(size=11,color="darkred"),row=2,col=1)
fig.update_layout(height=650, title_text="Survival by Molecular Subtype (Global)")
fig.update_yaxes(range=[-0.05,1.05])
fig.show()


In [ ]:
# Per-cancer-group subtype analysis
ct = pd.crosstab(samples["MOLECULAR_SUBTYPE"], samples["CANCER_GROUP"])
div = samples.groupby("CANCER_GROUP")["MOLECULAR_SUBTYPE"].nunique().sort_values(ascending=False)
print("Subtype diversity per cancer group:")
print(div)
rich = div[div>=5].index.tolist()
print(f"\nGroups with >=5 subtypes: {rich}")
for cg in rich[:3]:
    cgm = merged[merged["CANCER_GROUP"]==cg]
    top = cgm["MOLECULAR_SUBTYPE"].value_counts().head(5).index.tolist()
    if len(top)<2: continue
    fig = make_subplots(rows=2, cols=1,
        subplot_titles=(f"OS by Subtype -- {cg}",f"EFS by Subtype -- {cg}"),
        vertical_spacing=0.15)
    os_d2 = []; ef_d2 = []
    for j, st in enumerate(top):
        sub = cgm[cgm["MOLECULAR_SUBTYPE"]==st]
        os_s = sub[["OS_MONTHS","os_event"]].dropna()
        if len(os_s)>3:
            km = kaplan_meier(os_s["OS_MONTHS"], os_s["os_event"])
            fig = add_km(fig, km, st, colors[j%len(colors)])
            os_d2.append((os_s["OS_MONTHS"], os_s["os_event"]))
        ef_s = sub[["EFS_MONTHS","efs_event"]].dropna()
        if len(ef_s)>3:
            km = kaplan_meier(ef_s["EFS_MONTHS"], ef_s["efs_event"])
            fig.add_trace(go.Scatter(x=km["time"], y=km["survival"],
                mode="lines", name=st, line=dict(color=colors[j%len(colors)],width=2,shape="hv"),
                legendgroup=st, showlegend=False), row=2, col=1)
            ef_d2.append((ef_s["EFS_MONTHS"], ef_s["efs_event"]))
    if len(os_d2)>=2:
        fig.add_annotation(xref="paper",yref="paper",x=0.5,y=1.02,
            text=f"p={multi_logrank(os_d2):.4f}",showarrow=False,font=dict(size=10,color="darkblue"),row=1,col=1)
        fig.add_annotation(xref="paper",yref="paper",x=0.5,y=1.02,
            text=f"p={multi_logrank(ef_d2):.4f}",showarrow=False,font=dict(size=10,color="darkred"),row=2,col=1)
    fig.update_layout(height=550, title_text=f"Survival by Subtype -- {cg}")
    fig.update_yaxes(range=[-0.05,1.05])
    fig.show()


In [ ]:
print("Top 10 subtypes:")
print(merged["MOLECULAR_SUBTYPE"].value_counts().head(10))
div = merged.groupby("CANCER_GROUP")["MOLECULAR_SUBTYPE"].nunique().sort_values(ascending=False)
print("\nDiversity:", div)


## Step 14: Age at Diagnosis by Cancer Group

**Purpose:** Whether different cancer groups occur at different ages.

In [ ]:
patients = read_patients()
samples = read_samples()
merged = samples.merge(patients, on="PATIENT_ID", how="left", suffixes=("","_p"))
top8 = merged["CANCER_GROUP"].value_counts().head(8).index.tolist()
pd8 = merged[merged["CANCER_GROUP"].isin(top8)].dropna(subset=["AGE"])
fig = go.Figure()
for i, cg in enumerate(top8):
    sub = pd8[pd8["CANCER_GROUP"]==cg]["AGE"]
    fig.add_trace(go.Box(y=sub, name=cg, boxmean="sd",
        marker_color=px.colors.qualitative.Plotly[i]))
groups = [merged[merged["CANCER_GROUP"]==cg]["AGE"].dropna() for cg in top8]
stat, p_kw = kruskal(*groups)
fig.add_annotation(xref="paper",yref="paper",x=0.5,y=1.05,
    text=f"Kruskal-Wallis: H={stat:.2f}, p={p_kw:.4f}",
    showarrow=False, font=dict(size=12, color="darkred"))
fig.update_layout(title="Age at Diagnosis by Cancer Group",
    yaxis_title="Age (years)", height=500, xaxis_tickangle=-45)
fig.show()
if p_kw < 0.05:
    print("Post-hoc (Mann-Whitney, p<0.01):")
    for i, c1 in enumerate(top8):
        for j, c2 in enumerate(top8):
            if i>=j: continue
            g1 = merged[merged["CANCER_GROUP"]==c1]["AGE"].dropna()
            g2 = merged[merged["CANCER_GROUP"]==c2]["AGE"].dropna()
            if len(g1)>5 and len(g2)>5:
                _, p = mannwhitneyu(g1,g2,alternative="two-sided")
                if p<0.01:
                    print(f"  {c1:35s} vs {c2:35s}: p={p:.6f}")


## Step 15: Sex Balance by Cancer Group

**Purpose:** Whether certain cancer groups show sex bias.

In [ ]:
patients = read_patients()
samples = read_samples()
merged = samples.merge(patients, on="PATIENT_ID", how="left", suffixes=("","_p"))
top8 = merged["CANCER_GROUP"].value_counts().head(8).index.tolist()
pd8 = merged[merged["CANCER_GROUP"].isin(top8)].copy()
pd8["SEX"] = pd8["SEX"].fillna("Unknown")
ct = pd.crosstab(pd8["CANCER_GROUP"], pd8["SEX"])
fig = go.Figure()
for sex in ct.columns:
    fig.add_trace(go.Bar(name=sex, x=ct.index, y=ct[sex],
        text=ct[sex], textposition="inside"))
chi2, p, _, _ = chi2_contingency(ct)
fig.add_annotation(xref="paper",yref="paper",x=0.5,y=1.05,
    text=f"Chi2: x2={chi2:.2f}, p={p:.4f}",
    showarrow=False, font=dict(size=12, color="darkred"))
fig.update_layout(barmode="stack", title="Sex Distribution by Cancer Group",
    xaxis_tickangle=-45, height=450, yaxis_title="Count")
fig.show()


In [ ]:
ct = pd.crosstab(merged["CANCER_GROUP"], merged["SEX"])
chi2, p, dof, _ = chi2_contingency(ct.fillna(0))
print(f"Chi2: x2={chi2:.2f}, p={p:.4f}, df={dof}")


## Step 16: Purity by Cancer Group & Tumor Type

**Purpose:** Tumor purity differences across cancer groups and clinical states.

In [ ]:
patients = read_patients()
samples = read_samples()
samples = clean_tumor_fraction_ploidy(samples)
merged = samples.merge(patients, on="PATIENT_ID", how="left", suffixes=("","_p"))
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("TF by Cancer Group","TF by Tumor Type"))
top8 = merged["CANCER_GROUP"].value_counts().head(8).index.tolist()
for i, cg in enumerate(top8):
    sub = merged[merged["CANCER_GROUP"]==cg]["TUMOR_FRACTION"].dropna()
    if len(sub)>0:
        fig.add_trace(go.Box(y=sub, name=cg, boxmean="sd",
            marker_color=px.colors.qualitative.Plotly[i]), row=1, col=1)
top_tt = ["primary","metastatic","progression","recurrence"]
for tt in top_tt:
    sub = merged[merged["TUMOR_TYPE"]==tt]["TUMOR_FRACTION"].dropna()
    if len(sub)>0:
        fig.add_trace(go.Box(y=sub, name=tt, boxmean="sd",
            marker_color=px.colors.qualitative.Set2[top_tt.index(tt)]), row=1, col=2)
fig.update_layout(height=500, title_text="Tumor Fraction Analysis",
    yaxis_title="Fraction", xaxis_tickangle=-45, xaxis2_tickangle=-45)
fig.show()
g_cg = [merged[merged["CANCER_GROUP"]==cg]["TUMOR_FRACTION"].dropna() for cg in top8
        if len(merged[merged["CANCER_GROUP"]==cg]["TUMOR_FRACTION"].dropna())>5]
if len(g_cg)>=2:
    s, p = kruskal(*g_cg); print(f"KW (CG): H={s:.2f}, p={p:.4f}")
g_tt = [merged[merged["TUMOR_TYPE"]==tt]["TUMOR_FRACTION"].dropna() for tt in top_tt
        if len(merged[merged["TUMOR_TYPE"]==tt]["TUMOR_FRACTION"].dropna())>5]
if len(g_tt)>=2:
    s, p = kruskal(*g_tt); print(f"KW (TT): H={s:.2f}, p={p:.4f}")


## Step 17: Predisposition vs Outcome

**Purpose:** Whether known cancer predisposition affects survival or age of onset.

In [ ]:
patients = read_patients()
patients = clean_predispositions(patients)
patients = clean_os_status(patients)
patients = clean_efs_status(patients)
patients["has_pred"] = ~patients["CANCER_PREDISPOSITIONS"].isin(["No predisposition","Unknown"])
print(f"With predisposition: {patients["has_pred"].sum()} / {len(patients)}")
fig = make_subplots(rows=2, cols=2,
    subplot_titles=("OS by Predisposition","EFS by Predisposition","Age by Predisposition","OS Event"),
    specs=[[{"type":"scatter"},{"type":"scatter"}],[{"type":"box"},{"type":"bar"}]])
for i, (lab, has) in enumerate([("No",False),("Yes",True)]):
    sub = patients[patients["has_pred"]==has]
    os_s = sub[["OS_MONTHS","os_event"]].dropna()
    if len(os_s)>3:
        km = kaplan_meier(os_s["OS_MONTHS"], os_s["os_event"])
        fig = add_km(fig, km, lab, px.colors.qualitative.Set1[i])
    ef_s = sub[["EFS_MONTHS","efs_event"]].dropna()
    if len(ef_s)>3:
        km = kaplan_meier(ef_s["EFS_MONTHS"], ef_s["efs_event"])
        fig.add_trace(go.Scatter(x=km["time"], y=km["survival"],
            mode="lines", name=lab,
            line=dict(color=px.colors.qualitative.Set1[i],width=2,shape="hv"),
            legendgroup=lab, showlegend=False), row=1, col=2)
    age = sub["AGE"].dropna()
    fig.add_trace(go.Box(y=age, name=lab, boxmean="sd",
        marker_color=px.colors.qualitative.Set1[i]), row=2, col=1)
fig.update_layout(height=600, title_text="Predisposition vs Outcome")
fig.show()


In [ ]:
# Statistical tests
t = patients[patients["has_pred"]==True]
f = patients[patients["has_pred"]==False]
os_t = t[["OS_MONTHS","os_event"]].dropna()
os_f = f[["OS_MONTHS","os_event"]].dropna()
if len(os_t)>3 and len(os_f)>3:
    print(f"Log-rank OS: p={logrank_test(os_t["OS_MONTHS"],os_t["os_event"],os_f["OS_MONTHS"],os_f["os_event"]):.4f}")
ef_t = t[["EFS_MONTHS","efs_event"]].dropna()
ef_f = f[["EFS_MONTHS","efs_event"]].dropna()
if len(ef_t)>3 and len(ef_f)>3:
    print(f"Log-rank EFS: p={logrank_test(ef_t["EFS_MONTHS"],ef_t["efs_event"],ef_f["EFS_MONTHS"],ef_f["efs_event"]):.4f}")
at = t["AGE"].dropna(); af = f["AGE"].dropna()
if len(at)>3 and len(af)>3:
    s, p = mannwhitneyu(at, af, alternative="two-sided")
    print(f"MW Age: U={s:.0f}, p={p:.4f}, medians: {at.median():.1f} vs {af.median():.1f}")


## Step 18: CNS Region vs Cancer Group

**Purpose:** Anatomical distribution patterns of different cancer types.

In [ ]:
patients = read_patients()
samples = read_samples()
merged = samples.merge(patients, on="PATIENT_ID", how="left", suffixes=("","_p"))
ct = pd.crosstab(merged["CNS_REGION"], merged["CANCER_GROUP"])
tr = merged["CNS_REGION"].value_counts().head(8).index
tc = merged["CANCER_GROUP"].value_counts().head(10).index
cf = ct.loc[ct.index.intersection(tr), ct.columns.intersection(tc)]
cn = cf.div(cf.sum(axis=1), axis=0).fillna(0)
fig = go.Figure(data=go.Heatmap(
    z=cn.values, x=cn.columns, y=cn.index,
    text=cf.values, texttemplate="%{text}", textfont=dict(size=10),
    colorscale="Purples", colorbar=dict(title="Proportion"),
    hovertemplate="Region: %{y}<br>Group: %{x}<br>Count: %{text}<br>Prop: %{z:.2f}<extra></extra>"))
fig.update_layout(title="CNS Region x Cancer Group (Row-Normalized)",
    xaxis_tickangle=-45, height=550)
fig.show()


In [ ]:
print(f"CNS_REGION missing: {merged["CNS_REGION"].isna().sum()}")
print(merged["CNS_REGION"].value_counts().head(8))


## Step 19: Generate Summary Report

**Purpose:** A markdown summary of all findings.

In [ ]:
patients = read_patients()
samples = read_samples()
patients = clean_os_status(patients)
patients = clean_efs_status(patients)
patients = clean_predispositions(patients)
samples = clean_molecular_subtype(samples)
n_p = patients["PATIENT_ID"].nunique()
n_s = samples["SAMPLE_ID"].nunique()
n_cg = samples["CANCER_GROUP"].nunique()
a_m = patients["AGE"].median()
a_r = (patients["AGE"].min(), patients["AGE"].max())
p_m = (patients["SEX"]=="Male").mean()*100
p_f = (patients["SEX"]=="Female").mean()*100
n_pr = (patients["CANCER_PREDISPOSITIONS"]!="No predisposition").sum()
t_cg = samples["CANCER_GROUP"].value_counts().index[0]
t_cg_c = samples["CANCER_GROUP"].value_counts().iloc[0]
n_uc = (samples["MOLECULAR_SUBTYPE"]=="Unclassified").sum()
n_mt = samples["TUMOR_FRACTION"].isna().sum()
n_mtp = samples["TUMOR_PLOIDY"].isna().sum()
os_c = patients[["OS_MONTHS","os_event"]].dropna()
os_m = os_c["OS_MONTHS"].median()
report = f"""# PBTA_RNA Basic Clinical Summary

## Dataset Overview\n- Patients: {n_p}\n- Samples: {n_s}\n- Cancer Groups: {n_cg}\n- Most common: {t_cg} ({t_cg_c})\n\n## Demographics\n- Age: median {a_m:.1f}y, range {a_r[0]:.0f}-{a_r[1]:.0f}\n- Sex: {p_m:.1f}% M, {p_f:.1f}% F\n\n## Survival\n- Median OS: {os_m:.1f}mo\n- Patients w/ OS data: {len(os_c)}/{n_p}\n\n## Predispositions\n- Known predisposition: {n_pr}/{n_p} ({n_pr/n_p*100:.1f}%)\n\n## Subtypes\n- Unclassified: {n_uc}/{n_s} ({n_uc/n_s*100:.1f}%)\n- Distinct subtypes: {samples["MOLECULAR_SUBTYPE"].nunique()}\n\n## Tumor Purity\n- Missing TF: {n_mt}/{n_s} ({n_mt/n_s*100:.1f}%)\n- Missing TP: {n_mtp}/{n_s} ({n_mtp/n_s*100:.1f}%)\n"""
with open("/home/alon/menow_home_ass/basic_clinical_summary.md","w") as f:
    f.write(report)
print("Saved: basic_clinical_summary.md")
print(report)


## Step 20: Summary Table of All Figures

**Purpose:** Consolidated overview of every figure produced.

In [ ]:
figures = [
    (1,"Missingness Bar Plot","Bar","Patient column missingness",True),
    (2,"Interactive Age Histogram","Histogram","Age by SEX/RACE/ETHNICITY",True),
    (2,"Demographics Grid","Bar+Table","Sex, race, ethnicity, age stats",True),
    (3,"OS Status Pie","Pie","Deceased vs Living",True),
    (3,"EFS Binary+Detailed Bars","Bar","Event-free survival categories",True),
    (3,"OS KM Curve","KM","OS probability over time",True),
    (3,"EFS KM Curve","KM","EFS probability over time",True),
    (4,"Predisposition Prevalence","Bar","Top 15 syndromes",True),
    (4,"Predisposition Explorer","Multi","Age/sex/prevalence per syndrome",True),
    (5,"Sample Missingness Bar","Bar","Missing data in sample columns",True),
    (6,"Cancer Types Grid","Bar","Histology, group, region, type",True),
    (7,"Tumor Fraction Histogram","Histogram","Purity distribution",True),
    (7,"Tumor Ploidy Histogram","Histogram","Ploidy distribution",True),
    (7,"Fraction vs Ploidy Scatter","Scatter","Purity vs ploidy by group",True),
    (8,"Molecular Subtype Bar","Bar","Top 20 subtypes",True),
    (8,"Subtype x Group Heatmap","Heatmap","Subtype-group cross-tab",True),
    (9,"Sequencing Strategy Bar","Bar","Experiment strategies",True),
    (9,"RNA Library Bar","Bar","Library prep methods",True),
    ("9a","Groups per Patient Bar","Bar","Distinct CGs per patient",True),
    ("9a","Group Co-occurrence Heatmap","Heatmap","CG pair patterns",True),
    (11,"Samples per Patient Histogram","Histogram","Samples/patient",True),
    (12,"OS by Cancer Group KM","KM","OS by top 6 CGs",True),
    (12,"EFS by Cancer Group KM","KM","EFS by top 6 CGs",True),
    (13,"OS by Subtype KM","KM","OS by top 6 subtypes",True),
    (13,"EFS by Subtype KM","KM","EFS by top 6 subtypes",True),
    (13,"Subtype KM per Group","KM","Subtype survival in rich groups",True),
    (14,"Age by Group Boxplot","Box","Age distribution with KW test",True),
    (15,"Sex by Group Bar","Bar","Sex balance with chi2 test",True),
    (16,"TF by Group Boxplot","Box","Purity across groups",True),
    (16,"TF by Type Boxplot","Box","Purity by clinical state",True),
    (17,"Predisposition KM","KM","OS/EFS by pred status",True),
    (17,"Age by Predisposition Box","Box","Age by pred status",True),
    (18,"CNS x Cancer Group Heatmap","Heatmap","Anatomical distribution",True),
]
fd = pd.DataFrame(figures, columns=["Step","Title","Type","Insight","Interactive?"])
fd["Interactive?"] = fd["Interactive?"].map({True:"Yes",False:"No"})
print(f"Total figures: {len(fd)}\n")
print(fd.to_string(index=False))


In [ ]:
print("="*70)
print("NOTEBOOK COMPLETE -- All 20 steps implemented")
print("="*70)
steps = [
    (1,"Load & Profile Patient Data"),(2,"Patient Demographics"),(3,"Patient Survival Overview"),
    (4,"Cancer Predispositions"),(5,"Load & Profile Sample Data"),(6,"Sample Cancer Types"),
    (7,"Tumor Purity & Ploidy"),(8,"Molecular Subtype Landscape"),(9,"Sequencing Strategy"),
    ("9a","Multi-Cancer-Group Analysis"),(10,"Merge Patient + Sample Data"),(11,"Samples per Patient"),
    (12,"Survival by Cancer Group"),(13,"Survival by Molecular Subtype"),(14,"Age by Cancer Group"),
    (15,"Sex Balance by Cancer Group"),(16,"Purity by Group & Type"),
    (17,"Predisposition vs Outcome"),(18,"CNS Region vs Cancer Group"),
    (19,"Generate Summary Report"),(20,"Summary Table of All Figures"),
]
for s, n in steps:
    print(f"  Step {str(s):3s}: {n}")
